# An example of PyAges to calibrate LPM models on CFCs data

In [ ]:
import sys
from pathlib import Path
import os

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / 'pyproject.toml').exists() and (parent / 'pyages').exists():
        ROOT = parent
        break

print('CWD:', os.getcwd())
print('ROOT:', ROOT)
print('Has pyages:', (ROOT / 'pyages').exists())
print('Has data_core:', (ROOT / 'data_core').exists())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('sys.path[0]:', sys.path[0])


All files necessary for the code to work

In [ ]:
%matplotlib inline

In [ ]:
import copy
import sys
import os
import math

from pyages.concentrations import Concentrations
from pyages.concentrations.chronicles import export_concentration_chronicles
from pyages.config.paths import ROOT_DIRECTORY, ROOT_DIRECTORY_RESULTS, result_subdirectory
from pyages.config.runtime import DisplayOptions
from pyages.lpm import build_lpm

from pyages.calibration.utils.systematic_sampling import SystematicSampling
from pyages.calibration.problem import CalibrationProblem
from pyages.lpm.plotting.sample_diagnostics import plot_concentration_diagnostics, plot_parameter_diagnostics
import pyages.calibration.methods.simplex as csimp
import pyages.calibration.methods.metropolis_hastings as cMH



## Parameters

In [ ]:
# ---------------- CONCENTRATIONS DATA ------------------
# Concentration data
# dataset_name = "SSW_2008.txt"
# date = 2007
dataset_name = "SSW_2007.txt"
dataset_year = 2007
verbose = True

## Output Directory

In [ ]:
# ---------------- OUTPUT DIRECTORY ----------------------
directory_results = result_subdirectory(ROOT_DIRECTORY_RESULTS, "test_cases")
directory_results = result_subdirectory(directory_results, dataset_name)


## Display Options

Selection of the different types of outputs

In [ ]:
# ---- DISPLAY OPTIONS + ROOT OUTPUT DIRECTORY ------------
display = DisplayOptions()
display.text = True
display.figure = True
display.figure_close = False
display.figure_save = True
display.directory = directory_results 

In [ ]:
# --- SETUP AFFICHAGE ---
%matplotlib inline
import matplotlib.pyplot as plt
plt.ion()

display.figure = True
display.figure_save = False
display.figure_close = False


## Concentration data

Defined by the directory and file names. These should be modified to match the location of the data in hard-drive directories

<span style="color:red">*Directory should be entered here if needed; default is examples/natural/albuquerque/data under ROOT_DIRECTORY*</span>

In [ ]:
# Data Loading
data_dir = Path(ROOT_DIRECTORY) / "examples" / "natural" / "albuquerque" / "data"
filename = str(data_dir / dataset_name)
if verbose:
    print("Data file location: ", filename)
concentration_sampled = Concentrations.from_file(filename)
# Adds some percentage of uncertainty to the data
# concentration_sampled.set_relative_errors(error_concentrations)
concentration_sampled.display(display)
# Copy results to root directory
concentration_sampled.frame.to_csv(os.path.join(display.directory,"concentrations.txt"),sep='\t')

## LPM model

Defined by te LPM type and the directory where the characteristics of the LPM are given
Parameters are acceptable boundaries, initial values for calibration and Metropolis Hastings parameters 

In [ ]:
lpm_model_name = "dirac_double"
directory_lpm = os.path.join(ROOT_DIRECTORY, "data_core", "data_lpm")
print("parameters for the calibration are in directory:\n\t", directory_lpm)

## Reachable Concentrations

With a systematic sampling of the parameter space, displays the concentrations that can be reached with the chosen LPM

In [ ]:
resolution=10000
# ---------------- REACHABLE CONCENTRATIONS -------------
directory_cr = result_subdirectory(display.directory, "reachable_concentrations")
display_cr=copy.deepcopy(display)
display_cr.figure_save = False
display_cr.figure_close = False
display_cr.directory = directory_cr

cr = SystematicSampling(lpm_model_name, concentration_sampled.tracer_names(), 
                        date=concentration_sampled.frame["date"], 
                                                  sample_count=resolution, observations=concentration_sampled, display_options=display_cr)
cr.compute_concentrations()
cr.output()
cr.display_concentrations_with_data()
plt.show()


In [ ]:
# --- DEBUG BACKEND ---
import matplotlib.pyplot as plt
print(plt.get_backend())
print('figures:', plt.get_fignums())
plt.show()


## Calibration
### Parameters
Parameters for the Forward error propagation and Metropolis Hastings Methods 

In [ ]:
# ---------------- CALIBRATION PARAMETERS ----------------
run_calibration_simplex = True
run_calibration_metropolis_hastings = True
calibration_strategies = [None] * 2

# ---------------- FORWARD UNCERTAINTY QUANTIFICATION -----------------------------
calibration_strategies[0] = csimp.Simplex("forward_uncertainty_quantification",
                                            init_multiples_n=5,fuq_n=50)

# ---------------- METROPOLIS HASTINGS --------------------
# Method and Parameters  
mh_config = cMH.MHConfig(
    nstep=25000,
    prior_option=False,
    likelihood=True,
    monitor=True,
    display_traj=True,
)
calibration_strategies[1] = cMH.MetropolisHastings(config=mh_config)
calibration_strategies[1].proposal_step.define_by_value()


### Calibration per se

In [ ]:
lpm_results=[None]*2
# ---------------- CALIBRATION -------------
for i in range(len(calibration_strategies)):
    # Outputs of Interpration
    directory_calibration = result_subdirectory(display.directory, calibration_strategies[i].method)
    display.directory = directory_calibration
    # Calibration
    calib_basis = CalibrationProblem(concentration_sampled, lpm_model_name, display_options=display, lpm_directory=directory_lpm)
    calib_basis.prepare()
    lpm_results[i] = calibration_strategies[i].run(calib_basis)
    # Stores/Writes Results
    calibration_strategies[i].write_calibrated_lpm(lpm_results[i])

### Displays Results as graphics 
Distribution of parameters 
Distribution of concentrations
Relations between parameters (when number of parameters is larger than 2) 

In [ ]:
# ---------------- SYNTHETIC FIGURES --------------------
plot_parameter_diagnostics(lpm_results[0], self_method=calibration_strategies[0].method, lpm_reference=None,
                                               lpm_2nd=lpm_results[1],
                                               lpm_2nd_method=calibration_strategies[1].method,
                                               directory=None)
plt.show()
plot_concentration_diagnostics(lpm_results[0], self_method=calibration_strategies[0].method,
                                                   concentrations_reference=concentration_sampled,
                                                   lpm_2nd=lpm_results[1],
                                                   lpm_2nd_method=calibration_strategies[1].method,
                                                   directory=None)
plt.show()


### Objective function representation
Crosscuts when parameter number is larger than 3 

In [ ]:

# ------- OBJECTIVE FUNCTION -------------------------------
resolution = 50000

ss = SystematicSampling(
    lpm_model_name,
    concentration_sampled.tracer_names(),
    date=concentration_sampled.frame["date"],
    observations=concentration_sampled,
    sample_count=resolution,
    display_options=display,
    explore_objective=True,
    explore_reachable=False,
)
ss.compute_concentrations()
ss.objective_function_build()
ss.objective_function_display()
plt.show()


### Resulting concentration chronicles of tracers and models 
Models are represented by lines of different colors

In [ ]:
# ------------- CONCENTRATION OUTPUTS ----------------------
lpm=build_lpm(lpm_model_name, directory_lpm=directory_lpm)
export_concentration_chronicles([display.directory], lpm, display)

All results (figures and data) are avialable in the following directory

In [ ]:
print(display.directory)